In [ ]:
from IPython.display import display
import marimo as mo
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm
from watermark import watermark

from pylib._osf_fetch import osf_fetch
from pylib.theory._calc_mutator_fixprob_closed import (
    calc_mutator_fixprob_closed,
)
from pylib.viz._add_legend_entry import add_legend_entry
from pylib.viz._size_fixation_areaplot import size_fixation_areaplot

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-25T22:14:28.557250+00:00

Python implementation: CPython
Python version       : 3.10.22
IPython version      : 7.31.1

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 7.0.0-1014-aws
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

teeplot: 1.2.0
pandas : 2.2.3
scipy  : 1.14.1
seaborn: 0.13.2
polars : 1.9.0
marimo : 0.23.2
numpy  : 2.1.2

```

In [ ]:
teeplot_subdir = "cupy-5050-traits"
teeplot_subdir

'cupy-5050-traits'

In [ ]:
data = [
    {
        "url": "https://osf.io/duam2/download",
        "population structure": "well-mixed",
    },
    {
        "url": "https://osf.io/8q5v6/download",
        "population structure": "2D demes",
    },
    # {
    #     "url": "https://osf.io/s67d2/download",
    #     "population structure": "1D demes",
    # },
]
dfs = []
for item in tqdm(data):
    df = pd.read_parquet(osf_fetch(item["url"]))
    df["population structure"] = item["population structure"]
    df["available beneficial mutations"] = df["nBen"]
    df = pl.from_pandas(df)
    df = (
        df.with_columns(
            pl.when(pl.col("nRowSubgrid") == 0)
            .then(pl.col("nRow"))
            .otherwise(pl.col("nRowSubgrid"))
            .alias("nRowSubgrid"),
            pl.when(pl.col("nColSubgrid") == 0)
            .then(pl.col("nCol"))
            .otherwise(pl.col("nColSubgrid"))
            .alias("nColSubgrid"),
        )
        .with_columns(
            (pl.col("row").cast(pl.UInt64) // pl.col("nRowSubgrid")).alias(
                "rowGroup"
            ),
            (pl.col("col").cast(pl.UInt64) // pl.col("nColSubgrid")).alias(
                "colGroup"
            ),
        )
        .with_columns(
            (
                pl.col("colGroup")
                + (pl.col("colGroup").max() + 1) * pl.col("rowGroup")
            ).alias("group"),
        )
        .with_columns(
            (
                pl.col("trait value").replace_strict(
                    {0: "nonmutator", 1: "mutator"},
                )
            ).alias("genotype"),
        )
        .with_columns(
            (
                pl.col("nColSubgrid").cast(pl.UInt64)
                * pl.col("nRowSubgrid").cast(pl.UInt64)
                * pl.col("tilePopSize")
            ).alias("population size"),
        )
        .with_columns(
            (pl.col("trait count") / pl.col("tilePopSize")).alias(
                "fixation probability"
            ),
        )
    )
    assert (df["nCol"] > 0).all()
    assert (df["nRow"] > 0).all()
    assert (df["nColSubgrid"] > 0).all()
    assert (df["nRowSubgrid"] > 0).all()
    assert (df["tilePopSize"] > 0).all()
    assert (df["population size"] > 0).all()
    dfs.append(
        df.group_by(
            [
                "group",
                "genotype",
                "replicate",
                "population size",
                "population structure",
                "available beneficial mutations",
            ],
        ).agg(
            pl.col("fixation probability").mean(),
        ),
    )

  0%|          | 0/2 [00:00<?, ?it/s]

downloading https://osf.io/duam2/download -> /tmp/duam2
size: 25587641 bytes


 50%|█████     | 1/2 [00:03<00:03,  3.76s/it]

downloading https://osf.io/8q5v6/download -> /tmp/8q5v6
size: 34632554 bytes


100%|██████████| 2/2 [00:07<00:00,  3.85s/it]


In [ ]:
dfxs = pl.concat(dfs, how="vertical_relaxed").to_pandas()
dfxs["fixation\nprobability"] = dfxs["fixation probability"]
dfxs["population\nstructure"] = dfxs["population structure"]
dfxs["available\nbeneficial\nmutations"] = dfxs[
    "available beneficial mutations"
]

display(dfxs.describe())
display(dfxs.head())
display(dfxs.tail())

/tmp/marimo_5213/__marimo__cell_bkHC_.py:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  dfxs = pl.concat(dfs, how="vertical_relaxed").to_pandas()


group population size available beneficial mutations fixation probability fixationprobability availablebeneficialmutations count 1.195884e+06 1.195884e+06 1.195884e+06 1195884.0 1195884.0 1.195884e+06 mean 2.724031e+04 3.185400e+03 3.111111e+00 0.5 0.5 3.111111e+00 std 1.742655e+04 1.771294e+05 2.024542e+00 0.5 0.5 2.024542e+00 min 0.000000e+00 2.560000e+02 1.000000e+00 0.0 0.0 1.000000e+00 25% 1.181200e+04 2.560000e+02 2.000000e+00 0.0 0.0 2.000000e+00 50% 2.582950e+04 2.560000e+02 3.000000e+00 0.5 0.5 3.000000e+00 75% 4.243900e+04 2.560000e+02 4.000000e+00 1.0 1.0 4.000000e+00 max 5.904800e+04 1.511654e+07 8.000000e+00 1.0 1.0 8.000000e+00 group genotype replicate population size population structure available beneficial mutations fixation probability fixationprobability populationstructure availablebeneficialmutations 0 49970 mutator d73cf4ab-6b43-4a3d-82ce-b097fd78f7d0 256 well-mixed 4 1.0 1.0 well-mixed 4 1 24268 nonmutator 95950ec5-2db5-4d65-b148-ed7fe438ca5c 256 well-mixed 3 1.0 1.0 well-mixed 3 2 38128 nonmutator 752a1681-5103-4a55-873d-e1cae5263de1 256 well-mixed 1 1.0 1.0 well-mixed 1 3 12784 mutator cc9b3613-4dfa-47e9-81cf-088b34854044 256 well-mixed 2 0.0 0.0 well-mixed 2 4 21560 mutator cc9b3613-4dfa-47e9-81cf-088b34854044 256 well-mixed 2 0.0 0.0 well-mixed 2 group genotype replicate population size population structure available beneficial mutations fixation probability fixationprobability populationstructure availablebeneficialmutations 1195879 21572 mutator 6ce5037a-573a-4c41-8e61-806e34e037d7 256 2D demes 1 0.0 0.0 2D demes 1 1195880 32146 nonmutator e44fa74f-479f-43a7-b641-ed7c84bfb94d 256 2D demes 2 1.0 1.0 2D demes 2 1195881 40049 mutator e44fa74f-479f-43a7-b641-ed7c84bfb94d 256 2D demes 2 0.0 0.0 2D demes 2 1195882 3057 nonmutator 85dfbb2c-5154-4990-af53-01355004b11a 256 2D demes 8 0.0 0.0 2D demes 8 1195883 56474 mutator 65c1b039-1425-4d32-9d0b-f5e99bf1548d 256 2D demes 3 0.0 0.0 2D demes 3

In [ ]:
apn = int(dfxs["population size"].min())  # agents per node
apn

256

In [ ]:
# math model predictions (see pylib/theory), for well-mixed populations
# and for a 2D lattice of 256-agent demes
model_N = np.geomspace(
    dfxs["population size"].min(), dfxs["population size"].max(), 60
)
model_xy = {
    float(K): [
        (
            model_N.tolist(),
            [
                calc_mutator_fixprob_closed(N, K, deme_size=deme_size)
                for N in model_N
            ],
        )
        for deme_size in (None, 256)
    ]
    for K in sorted(dfxs["available beneficial mutations"].unique())
    if K <= 4
}

/home/runner/work/hypermutator-dynamics/hypermutator-dynamics/bindle/pylib/theory/_calc_drift_fixprob.py:15: RuntimeWarning: overflow encountered in expm1
  return float(np.expm1(a * x0) / np.expm1(a))


In [ ]:
for _errorbar in "sd", "se", "ci", None:
    with tp.teed(
        size_fixation_areaplot,
        fixprobs_df=dfxs[dfxs["available beneficial mutations"] <= 4],
        x="population size",
        y="fixation probability",
        hue="genotype",
        hue_order=["nonmutator", "mutator"],
        col="available\nbeneficial\nmutations",
        row="population\nstructure",
        style="genotype",
        style_order=["nonmutator", "mutator"],
        errorbar=_errorbar,
        height=1.65,
        facet_kws=dict(legend_out=False, margin_titles=True),
        markers=True,
        teeplot_subdir=teeplot_subdir,
    ) as _teed:
        # teed.figure.suptitle(f"n={apn} agents per node", x=0.63, y=1.06)
        _teed.set_titles(col_template="{col_var} = {col_name:.0f}")
        _teed.set_titles(row_template="{row_name}")
        _teed.set_ylabels("fixation\nprobability")
        _teed.tight_layout()
        sns.move_legend(
            _teed,
            "lower center",
            bbox_to_anchor=(0.5, -0.07),
            frameon=False,
            ncol=2,
            title=None,
        )
        for _ax in _teed.axes.flat:
            _ax.set_xticks([1e3, 1e5, 1e7])
        _teed.tight_layout()

/home/runner/work/hypermutator-dynamics/hypermutator-dynamics/bindle/pylib/viz/_size_fixation_areaplot.py:24: UserWarning: The palette list has more values (10) than needed (2), which may not be intended.
  g = sns.relplot(


teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=sd+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=sd+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.png


/home/runner/work/hypermutator-dynamics/hypermutator-dynamics/bindle/pylib/viz/_size_fixation_areaplot.py:24: UserWarning: The palette list has more values (10) than needed (2), which may not be intended.
  g = sns.relplot(


teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=se+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=se+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.png


/home/runner/work/hypermutator-dynamics/hypermutator-dynamics/bindle/pylib/viz/_size_fixation_areaplot.py:24: UserWarning: The palette list has more values (10) than needed (2), which may not be intended.
  g = sns.relplot(


teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=ci+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=ci+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.png


/home/runner/work/hypermutator-dynamics/hypermutator-dynamics/bindle/pylib/viz/_size_fixation_areaplot.py:24: UserWarning: The palette list has more values (10) than needed (2), which may not be intended.
  g = sns.relplot(


teeplots/cupy-5050-traits/col=available-beneficial-mutations+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+hue=genotype+row=population-structure+style=genotype+viz=size-fixation-areaplot+x=population-size+y=fixation-probability+ext=.png


In [ ]:
for _errorbar in "sd", "se", "ci", None:
    with tp.teed(
        sns.relplot,
        data=dfxs[
            (dfxs["available beneficial mutations"] <= 4)
            & (dfxs["genotype"] == "mutator")
        ],
        x="population size",
        y="fixation probability",
        hue="population\nstructure",
        col="available\nbeneficial\nmutations",
        style="population\nstructure",
        aspect=0.7,
        errorbar=_errorbar,
        height=2,
        facet_kws=dict(legend_out=False, margin_titles=True),
        kind="line",
        markers=True,
        palette="Dark2",
        teeplot_subdir=teeplot_subdir,
    ) as _teed:
        _teed.set(xscale="log")
        _teed.set(ylim=(0, 1))
        for _K, _ax in _teed.axes_dict.items():
            for _curve in model_xy[_K]:
                _ax.plot(*_curve, color="black", linestyle="--")
        _teed.set_titles(col_template="{col_var} = {col_name:.0f}")
        _teed.set_titles(row_template="{row_name}")
        _teed.set_ylabels("mutator\nfixation\nprobability")
        _teed.tight_layout()
        sns.move_legend(
            _teed,
            "lower center",
            bbox_to_anchor=(0.5, -0.09),
            frameon=False,
            ncol=2,
            title=None,
        )
        add_legend_entry(_teed, "analytic approximation")
        for _ax in _teed.axes.flat:
            _ax.set_xticks([1e3, 1e5, 1e7])
            _ax.axhline(0.5, color="black", linestyle="--")
        _teed.tight_layout()

teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=sd+hue=population-structure+kind=line+palette=dark2+style=population-structure+viz=relplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=sd+hue=population-structure+kind=line+palette=dark2+style=population-structure+viz=relplot+x=population-size+y=fixation-probability+ext=.png
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=se+hue=population-structure+kind=line+palette=dark2+style=population-structure+viz=relplot+x=population-size+y=fixation-probability+ext=.pdf
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=se+hue=population-structure+kind=line+palette=dark2+style=population-structure+viz=relplot+x=population-size+y=fixation-probability+ext=.png
teeplots/cupy-5050-traits/col=available-beneficial-mutations+errorbar=ci+hue=population-structure+kind=line+palette=dark2+style=population-structure+viz=rel

In [ ]:
groupby = dfxs[
    (dfxs["available beneficial mutations"] <= 4)
    & (dfxs["genotype"] == "mutator")
].groupby(["available beneficial mutations", "population size"])

results = []
for (beneficial_mut, pop_size), group in groupby:
    (g1, grp1), (g2, grp2) = group.groupby("population\nstructure")
    group_2D = grp1
    group_well_mixed = grp2

    n_g1 = grp1.shape[0]
    n_g2 = grp2.shape[0]

    stat, p_val = scipy_stats.mannwhitneyu(
        grp1["fixation probability"],
        grp2["fixation probability"],
        alternative="two-sided",
    )

    results.append(
        {
            "available beneficial mutations": beneficial_mut,
            "population size": pop_size,
            f"n {g1}": n_g1,
            f"n {g2}": n_g2,
            "U statistic": stat,
            "p-value": p_val,
            "Bonferroni corrected p-value": min(p_val * len(groupby), 1.0),
        }
    )

results_table = pd.DataFrame(results)
results_table

,available beneficial mutations,population size,n 2D demes,n well-mixed,U statistic,p-value,Bonferroni corrected p-value
0,1,256,59049,59049,1.743392e+09,1.000000,1.000000
1,1,2304,6561,6561,2.156273e+07,0.750414,1.000000
2,1,20736,729,729,2.471310e+05,0.000231,0.005542
3,1,186624,81,81,2.916000e+03,0.093565,1.000000
4,1,1679616,9,9,4.050000e+01,1.000000,1.000000
...,...,...,...,...,...,...,...
19,4,2304,6561,6561,2.173659e+07,0.065305,1.000000
20,4,20736,729,729,2.657205e+05,1.000000,1.000000
21,4,186624,81,81,3.280500e+03,1.000000,1.000000
22,4,1679616,9,9,4.050000e+01,1.000000,1.000000
